# Analisis de los clustres más interesantes tras realizar todas las pruebas 

## GMM estandarazado Sin transformaciones
'tiempo_seg_cond4', 'ratio_albumina_creatinina' K=5

In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from kmodes.kprototypes import KPrototypes
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import unicodedata
import re
import gower
import plotly.express as px
import kmedoids # Nueva librería para PAM
from sklearn.cluster import AgglomerativeClustering
from sklearn.mixture import GaussianMixture

from scipy.cluster.hierarchy import dendrogram, linkage
import scipy.spatial.distance as ssd
import prince



In [2]:
trainset_no_ohe = pd.read_csv("trainset_noOHE.csv")

In [3]:
trainset_no_ohe.head(10)

,edad_an,tamano_hogar,psu,ratio_pobreza,tiempo_seg_cond1,tiempo_seg_cond2,tiempo_seg_cond3,tiempo_seg_cond4,pulso,peso_kg,...,elegibilidad_balance,convulsiones,caidas,fracturas,medicacion_osteoporosis,aprueba_cond4,tamano_manguito_pa,estado_elastografia,tipo_sonda_elasto,raza_etnia
0,-0.188751,1.575278,1.021598,0.984181,0.144067,0.245103,0.162136,0.810059,-0.438261,1.282766,...,1,0,0,0,0,1,4.0,1,b'XL',3.0
1,0.575265,0.208042,1.021598,-0.578577,0.144067,0.245103,0.162136,0.810059,0.158633,1.859173,...,1,0,0,0,0,1,4.0,1,b'XL',3.0
2,-0.049839,0.208042,1.021598,1.226568,0.144067,0.245103,0.162136,0.810059,-0.352991,0.254454,...,0,0,0,1,0,1,4.0,1,b'M',3.0
3,-0.605487,1.575278,-0.978858,0.021012,0.144067,0.245103,0.162136,0.810059,0.840797,-0.086779,...,1,0,0,0,0,1,3.0,1,b'M',2.0
4,1.200370,0.208042,1.021598,0.250642,0.144067,0.245103,0.162136,-1.268443,2.716750,-0.294286,...,1,0,0,0,0,0,3.0,1,b'M',6.0
5,-1.508416,-0.475576,-0.978858,1.226568,0.144067,0.245103,0.162136,0.810059,-0.182450,-1.520881,...,1,0,0,0,0,1,3.0,1,b'M',7.0
6,1.408738,-1.159194,-0.978858,-0.412733,0.144067,0.245103,0.162136,0.810059,-1.802591,0.120728,...,1,0,0,0,0,1,4.0,1,b'XL',3.0
7,-1.091680,1.575278,1.021598,-1.292981,0.144067,0.245103,0.162136,0.810059,-1.120426,0.019280,...,1,0,0,0,0,1,3.0,1,b'M',3.0
8,0.366897,0.891660,-0.978858,1.226568,0.144067,0.245103,0.162136,0.054240,0.158633,0.480406,...,1,0,0,0,0,1,4.0,1,b'XL',3.0
9,-0.605487,-1.159194,-0.978858,1.226568,0.144067,0.245103,0.162136,-1.835307,-1.717320,-0.796913,...,1,0,0,0,0,0,3.0,1,b'M',3.0


In [4]:
print(trainset_no_ohe.shape)

(1919, 62)


In [5]:
# 1. Cargar diccionario



ruta_diccionario = "../data/Diccionario_TFM_CompletoMF.xlsx"
diccionario = pd.read_excel(ruta_diccionario)

def normalizar_texto(x):
    x = str(x).strip().lower()
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("utf-8")
    return x

def limpiar_nombre(col):
    col = col.lower()
    col = col.replace(" ", "_")
    col = re.sub(r'[^a-z0-9_]', '', col)
    return col

diccionario["Tipo de Variable"] = diccionario["Tipo de Variable"].apply(normalizar_texto)

cols_categoricas = diccionario.loc[
    diccionario["Tipo de Variable"].str.contains("categ", na=False), "Qué es"
].tolist()
var_categoricas = [col for col in cols_categoricas if col in trainset_no_ohe.columns]

cols_numericas = diccionario.loc[
    diccionario["Tipo de Variable"].str.contains("num", na=False), "Qué es"
].tolist()
var_numericas = [col for col in cols_numericas if col in trainset_no_ohe.columns]

trainset_no_ohe[var_numericas] = trainset_no_ohe[var_numericas].astype('float64')
trainset_no_ohe[var_categoricas] = trainset_no_ohe[var_categoricas].astype('object')

print(f"Número de columnas categóricas: {len(var_categoricas)}")
print(var_categoricas)

print(f"Número de columnas numéricas: {len(var_numericas)}")
print(var_numericas)

Número de columnas categóricas: 16
['genero', 'periodo_examen', 'pais_nacimiento', 'toma_suplementos', 'toma_antiacidos', 'estado_examen_balance', 'elegibilidad_balance', 'convulsiones', 'caidas', 'fracturas', 'medicacion_osteoporosis', 'aprueba_cond4', 'tamano_manguito_pa', 'estado_elastografia', 'tipo_sonda_elasto', 'raza_etnia']
Número de columnas numéricas: 46
['edad_an', 'tamano_hogar', 'psu', 'ratio_pobreza', 'tiempo_seg_cond1', 'tiempo_seg_cond2', 'tiempo_seg_cond3', 'tiempo_seg_cond4', 'pulso', 'peso_kg', 'altura_cm', 'imc', 'largo_pierna_superior_cm', 'largo_brazo_superior_cm', 'medidas_validas_elasto', 'intentos_totales_elasto', 'rigidez_mediana_kpa', 'rigidez_iqr', 'ratio_iqr_mediana', 'cap_mediana_db_m', 'cap_iqr', 'albumina_orina_mg_l', 'creatinina_orina_umol_l', 'ratio_albumina_creatinina', 'peso_flebotomia', 'proteina_c_reactiva_mg_l', 'leucocitos_totales', 'linfocitos_abs', 'monocitos_abs', 'neutrofilos_abs', 'eosinofilos_abs', 'eritrocitos_totales', 'hemoglobina_g_dl',

In [6]:
var_num = trainset_no_ohe.select_dtypes(include=['number'])
var_num.shape

(1919, 46)

In [7]:
columnas = ['tiempo_seg_cond4', 'ratio_albumina_creatinina']
X = var_num[columnas].copy()

# 1. Crear y ajustar el modelo
gmm = GaussianMixture(n_components=5, random_state=42)
gmm.fit(X)

# 2. Predecir a qué clúster pertenece cada observación
labels = gmm.predict(X)

# 3. Calcular y mostrar el Silhouette Score
# Se calcula usando las variables originales (X) y las etiquetas predichas (labels)
sil_score = silhouette_score(X, labels)
print("--- Evaluación del Modelo ---")
print(f"Silhouette Score: {sil_score:.4f}\n")

# 4. Guardar las etiquetas en el dataframe
X['cluster'] = labels

# 5. Mostrar el conteo de filas por clúster
conteos = X['cluster'].value_counts().sort_index()
print("--- Número de filas por clúster ---")
for cluster_id, cantidad in conteos.items():
    print(f"Clúster {cluster_id}: {cantidad} filas")
print("-----------------------------------\n")

X['cluster'] = X['cluster'].astype(str)

--- Evaluación del Modelo ---
Silhouette Score: 0.6082

--- Número de filas por clúster ---
Clúster 0: 952 filas
Clúster 1: 363 filas
Clúster 2: 364 filas
Clúster 3: 51 filas
Clúster 4: 189 filas
-----------------------------------



In [9]:
df_train_target = pd.read_csv("../trainset_target.csv")


In [10]:
df_train_target1=df_train_target.copy()

In [11]:
df_train_target1.shape

(1919, 1)

In [12]:
df_train_target1['cluster_gmm'] = labels
print(df_train_target1.head())

# (Opcional) Ver la relación rápida entre los clústeres y la hipertensión


   hipertension  cluster_gmm
0             1            0
1             0            0
2             1            0
3             1            0
4             1            3


In [13]:
print("\n--- Relación: Clúster vs Hipertensión ---")
print(pd.crosstab(df_train_target1['cluster_gmm'], df_train_target1['hipertension']))


--- Relación: Clúster vs Hipertensión ---
hipertension    0    1
cluster_gmm           
0             565  387
1             190  173
2             205  159
3              11   40
4              78  111


### K-means para las variables de interés clinico K=2

    'ratio_albumina_creatinina',
    'plomo_sangre_umol_l',
    'proteina_c_reactiva_mg_l',
    'rigidez_mediana_kpa',
    'trigliceridos_mmol_l',
    'imc'

In [14]:
# 1. Seleccionar las variables de interés del nuevo dataframe
columnas = [
    'ratio_albumina_creatinina',
    'plomo_sangre_umol_l',
    'proteina_c_reactiva_mg_l',
    'rigidez_mediana_kpa',
    'trigliceridos_mmol_l',
    'imc'
]
X2 = var_num[columnas].copy()

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans.fit(X2)

# 3. Predecir a qué clúster pertenece cada observación
labels2 = kmeans.predict(X2)

# 4. Guardar las etiquetas temporalmente para el conteo y la gráfica
X2['cluster'] = labels2

# Mostrar el tamaño de los clústeres
conteos = X2['cluster'].value_counts().sort_index()
print("--- Número de filas por clúster (K-Means) ---")
for cluster_id, cantidad in conteos.items():
    print(f"Clúster {cluster_id}: {cantidad} filas")
print("---------------------------------------------\n")

# 5. Calcular el Silhouette Score (¡Excluyendo la etiqueta para no inflarlo!)
score = silhouette_score(X2.drop(columns=['cluster']), labels2)
print(f"El Silhouette Score para K-Means con k=2 es: {score:.4f}\n")

# 6. Preparar para la visualización (evitar degradado de color)
X2['cluster'] = X2['cluster'].astype(str)


--- Número de filas por clúster (K-Means) ---
Clúster 0: 1485 filas
Clúster 1: 434 filas
---------------------------------------------

El Silhouette Score para K-Means con k=2 es: 0.4313



In [15]:
df_train_target2=df_train_target.copy()

In [16]:
df_train_target2['clusters'] = labels2
print(df_train_target2.head())

# (Opcional) Ver la relación rápida entre los clústeres y la hipertensión


   hipertension  clusters
0             1         1
1             0         1
2             1         0
3             1         0
4             1         1


In [17]:
print("\n--- Relación: Clúster vs Hipertensión ---")
print(pd.crosstab(df_train_target2['clusters'], df_train_target2['hipertension']))


--- Relación: Clúster vs Hipertensión ---
hipertension    0    1
clusters              
0             874  611
1             175  259


### K-means para las variables de interés clinico K=2 y K3

  'ratio_albumina_creatinina',
    'proteina_c_reactiva_mg_l',
    'trigliceridos_mmol_l',
    'imc'

In [18]:
# 1. Seleccionar las variables de interés del nuevo dataframe
columnas = [
    'ratio_albumina_creatinina',
    'proteina_c_reactiva_mg_l',
    'trigliceridos_mmol_l',
    'imc'
]
X3 = var_num[columnas].copy()

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans.fit(X3)

# 3. Predecir a qué clúster pertenece cada observación
labels3 = kmeans.predict(X3)

# 4. Guardar las etiquetas temporalmente para el conteo y la gráfica
X3['cluster'] = labels3

# Mostrar el tamaño de los clústeres
conteos = X3['cluster'].value_counts().sort_index()
print("--- Número de filas por clúster (K-Means) ---")
for cluster_id, cantidad in conteos.items():
    print(f"Clúster {cluster_id}: {cantidad} filas")
print("---------------------------------------------\n")

# 5. Calcular el Silhouette Score (¡Excluyendo la etiqueta para no inflarlo!)
score = silhouette_score(X3.drop(columns=['cluster']), labels3)
print(f"El Silhouette Score para K-Means con k=2 es: {score:.4f}\n")

# 6. Preparar para la visualización (evitar degradado de color)
X3['cluster'] = X3['cluster'].astype(str)


--- Número de filas por clúster (K-Means) ---
Clúster 0: 1444 filas
Clúster 1: 475 filas
---------------------------------------------

El Silhouette Score para K-Means con k=2 es: 0.4629



In [19]:
df_train_target3=df_train_target.copy()

In [20]:
df_train_target3['clusters'] = labels3
print(df_train_target3.head())


   hipertension  clusters
0             1         1
1             0         1
2             1         0
3             1         0
4             1         1


In [21]:
print("\n--- Relación: Clúster vs Hipertensión ---")
print(pd.crosstab(df_train_target3['clusters'], df_train_target3['hipertension']))


--- Relación: Clúster vs Hipertensión ---
hipertension    0    1
clusters              
0             851  593
1             198  277


K-means k=3

In [22]:
# 1. Seleccionar las variables de interés del nuevo dataframe
columnas = [
    'ratio_albumina_creatinina',
    'proteina_c_reactiva_mg_l',
    'trigliceridos_mmol_l',
    'imc'
]
X4 = var_num[columnas].copy()

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(X4)

# 3. Predecir a qué clúster pertenece cada observación
labels4 = kmeans.predict(X4)

# 4. Guardar las etiquetas temporalmente para el conteo y la gráfica
X4['cluster'] = labels4

# Mostrar el tamaño de los clústeres
conteos = X4['cluster'].value_counts().sort_index()
print("--- Número de filas por clúster (K-Means) ---")
for cluster_id, cantidad in conteos.items():
    print(f"Clúster {cluster_id}: {cantidad} filas")
print("---------------------------------------------\n")

# 5. Calcular el Silhouette Score (¡Excluyendo la etiqueta para no inflarlo!)
score = silhouette_score(X4.drop(columns=['cluster']), labels4)
print(f"El Silhouette Score para K-Means con k=2 es: {score:.4f}\n")

# 6. Preparar para la visualización (evitar degradado de color)
X4['cluster'] = X4['cluster'].astype(str)


--- Número de filas por clúster (K-Means) ---
Clúster 0: 538 filas
Clúster 1: 14 filas
Clúster 2: 1367 filas
---------------------------------------------

El Silhouette Score para K-Means con k=2 es: 0.4359



In [23]:
df_train_target4=df_train_target.copy()

In [24]:
df_train_target4['clusters'] = labels4
print(df_train_target4.head())


   hipertension  clusters
0             1         0
1             0         0
2             1         2
3             1         2
4             1         1


In [25]:
print("\n--- Relación: Clúster vs Hipertensión ---")
print(pd.crosstab(df_train_target4['clusters'], df_train_target4['hipertension']))


--- Relación: Clúster vs Hipertensión ---
hipertension    0    1
clusters              
0             234  304
1               3   11
2             812  555


In [26]:
#comando de LIMPIEZA DE KERNEL PARA EVITAR QUE SE MEZCLEN VARIBALES DE DISTINTAS PRUEBAS
%reset -f
%reset -f in out dhist

Flushing input history
Flushing output cache (0 entries)
Flushing directory history


-----------------------------------------------------
-----------------------------------------------------

## K-Means estandarazado y con transformaciones logaritmica y raiz cuadrada.
'tiempo_seg_cond4', 'ancho_distribucion_eritrocitos' K=3

In [27]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from kmodes.kprototypes import KPrototypes
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import unicodedata
import re
import gower
from sklearn.preprocessing import StandardScaler
import kmedoids # Nueva librería para PAM
from sklearn.cluster import AgglomerativeClustering
from sklearn.mixture import GaussianMixture
import plotly.express as px
from scipy.cluster.hierarchy import dendrogram, linkage
import scipy.spatial.distance as ssd
import prince



In [28]:
trainset = pd.read_csv("train_imputado.csv")

In [29]:
trainset.head(10)

,edad_an,tamano_hogar,psu,ratio_pobreza,tiempo_seg_cond1,tiempo_seg_cond2,tiempo_seg_cond3,tiempo_seg_cond4,pulso,peso_kg,...,elegibilidad_balance,convulsiones,caidas,fracturas,medicacion_osteoporosis,aprueba_cond4,tamano_manguito_pa,estado_elastografia,tipo_sonda_elasto,raza_etnia
0,46.0,5.0,2.0,4.62,15.0,15.0,30.0,30.0,66.0,111.6,...,1,0,0,0,0,1,4.0,1,b'XL',3.0
1,57.0,3.0,2.0,2.17,15.0,15.0,30.0,30.0,73.0,124.1,...,1,0,0,0,0,1,4.0,1,b'XL',3.0
2,48.0,3.0,2.0,5.00,15.0,15.0,30.0,30.0,67.0,89.3,...,0,0,0,1,0,1,4.0,1,b'M',3.0
3,40.0,5.0,1.0,3.11,15.0,15.0,30.0,30.0,81.0,81.9,...,1,0,0,0,0,1,3.0,1,b'M',2.0
4,66.0,3.0,2.0,3.47,15.0,15.0,30.0,8.0,103.0,77.4,...,1,0,0,0,0,0,3.0,1,b'M',6.0
5,27.0,2.0,1.0,5.00,15.0,15.0,30.0,30.0,69.0,50.8,...,1,0,0,0,0,1,3.0,1,b'M',7.0
6,69.0,1.0,1.0,2.43,15.0,15.0,30.0,30.0,50.0,86.4,...,1,0,0,0,0,1,4.0,1,b'XL',3.0
7,33.0,5.0,2.0,1.05,15.0,15.0,30.0,30.0,58.0,84.2,...,1,0,0,0,0,1,3.0,1,b'M',3.0
8,54.0,4.0,1.0,5.00,15.0,15.0,30.0,22.0,73.0,94.2,...,1,0,0,0,0,1,4.0,1,b'XL',3.0
9,40.0,1.0,1.0,5.00,15.0,15.0,30.0,2.0,51.0,66.5,...,1,0,0,0,0,0,3.0,1,b'M',3.0


In [30]:
# 1. Cargar diccionario



ruta_diccionario = "../data/Diccionario_TFM_CompletoMF.xlsx"
diccionario = pd.read_excel(ruta_diccionario)

def normalizar_texto(x):
    x = str(x).strip().lower()
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("utf-8")
    return x

def limpiar_nombre(col):
    col = col.lower()
    col = col.replace(" ", "_")
    col = re.sub(r'[^a-z0-9_]', '', col)
    return col

diccionario["Tipo de Variable"] = diccionario["Tipo de Variable"].apply(normalizar_texto)

cols_categoricas = diccionario.loc[
    diccionario["Tipo de Variable"].str.contains("categ", na=False), "Qué es"
].tolist()
var_categoricas = [col for col in cols_categoricas if col in trainset.columns]

cols_numericas = diccionario.loc[
    diccionario["Tipo de Variable"].str.contains("num", na=False), "Qué es"
].tolist()
var_numericas = [col for col in cols_numericas if col in trainset.columns]

trainset[var_numericas] = trainset[var_numericas].astype('float64')
trainset[var_categoricas] = trainset[var_categoricas].astype('object')

print(f"Número de columnas categóricas: {len(var_categoricas)}")
print(var_categoricas)

print(f"Número de columnas numéricas: {len(var_numericas)}")
print(var_numericas)

Número de columnas categóricas: 16
['genero', 'periodo_examen', 'pais_nacimiento', 'toma_suplementos', 'toma_antiacidos', 'estado_examen_balance', 'elegibilidad_balance', 'convulsiones', 'caidas', 'fracturas', 'medicacion_osteoporosis', 'aprueba_cond4', 'tamano_manguito_pa', 'estado_elastografia', 'tipo_sonda_elasto', 'raza_etnia']
Número de columnas numéricas: 46
['edad_an', 'tamano_hogar', 'psu', 'ratio_pobreza', 'tiempo_seg_cond1', 'tiempo_seg_cond2', 'tiempo_seg_cond3', 'tiempo_seg_cond4', 'pulso', 'peso_kg', 'altura_cm', 'imc', 'largo_pierna_superior_cm', 'largo_brazo_superior_cm', 'medidas_validas_elasto', 'intentos_totales_elasto', 'rigidez_mediana_kpa', 'rigidez_iqr', 'ratio_iqr_mediana', 'cap_mediana_db_m', 'cap_iqr', 'albumina_orina_mg_l', 'creatinina_orina_umol_l', 'ratio_albumina_creatinina', 'peso_flebotomia', 'proteina_c_reactiva_mg_l', 'leucocitos_totales', 'linfocitos_abs', 'monocitos_abs', 'neutrofilos_abs', 'eosinofilos_abs', 'eritrocitos_totales', 'hemoglobina_g_dl',

In [31]:
var_num = trainset.select_dtypes(include=['number'])
var_num.shape

(1919, 46)

obtenemos el skew para cada variable: medida estadística que describe la falta de simetría en la distribución de tus datos alrededor de su media.

In [32]:
asimetria = var_num.skew().sort_values(ascending=False)

print("--- Nivel de Asimetría (Skewness) de las Variables ---")
print("Valores ideales: entre -1 y 1. Valores > 1 o < -1 indican asimetría severa.\n")
print(asimetria)

--- Nivel de Asimetría (Skewness) de las Variables ---
Valores ideales: entre -1 y 1. Valores > 1 o < -1 indican asimetría severa.

plomo_sangre_umol_l               19.482658
ratio_iqr_mediana                 18.586299
rigidez_iqr                       18.500563
ratio_albumina_creatinina         14.215589
albumina_orina_mg_l               11.529683
rigidez_mediana_kpa                7.974745
proteina_c_reactiva_mg_l           6.952438
trigliceridos_mmol_l               6.916607
cadmio_sangre_nmol_l               5.434901
eosinofilos_abs                    5.417549
selenio_sangre_umol_l              5.352683
ancho_distribucion_eritrocitos     5.211623
medidas_validas_elasto             5.064329
mercurio_sangre_nmol_l             4.743745
intentos_totales_elasto            1.972235
cap_iqr                            1.838814
peso_flebotomia                    1.829170
manganeso_sangre_nmol_l            1.501256
neutrofilos_abs                    1.391176
creatinina_orina_umol_l         

se puede observar que para ciertas variables la transformación logaritmica no será sido del todo útil esto es debido a:

1. medidas como plomo_sangre_umol_l son muy pequeñas y la la función log1p(x) uma 1 antes de sacar el logaritmo por tanto si se le suma 1 a un valor muy pequeño es practicamnete 1.

Para estos valores se aplica la transformación de raiz cuadrada.

In [33]:
skew_actual = var_num.skew()
var_num_log = var_num.copy()

# 2. Filtrar las variables con asimetría severa
variables_sesgadas = skew_actual[skew_actual.abs() > 1].index.tolist()

print("--- VARIABLES DETECTADAS PARA TRANSFORMACIÓN ---")
for col in variables_sesgadas:
    print(f"- {col} (Skew original: {skew_actual[col]:.2f})")

# Definimos las variables rebeldes que necesitan raíz cuadrada en lugar de logaritmo
variables_raiz = ['plomo_sangre_umol_l']

print("\n--- APLICANDO TRANSFORMACIONES ---")
# 3. Aplicar la transformación (Logaritmo o Raíz Cuadrada)
for col in variables_sesgadas:
    # Si la variable está en nuestra lista de excepciones, aplicamos raíz cuadrada
    if col in variables_raiz:
        var_num_log[col] = np.sqrt(var_num_log[col])
        print(f"✓ Raíz cuadrada aplicada a: {col}")
    else:
        # Para el resto, verificamos que no haya valores negativos antes de aplicar el logaritmo
        if (var_num_log[col] < 0).any():
            print(f"⚠️ Advertencia: {col} tiene valores negativos. No se aplicó logaritmo.")
        else:
            var_num_log[col] = np.log1p(var_num_log[col])
            # print(f"✓ Logaritmo aplicado a: {col}") # Opcional: descomentar para ver el progreso

print("\nTransformaciones aplicadas con éxito.")

# 4. Verificar los nuevos niveles de asimetría para confirmar la mejora
print("\n--- NUEVA ASIMETRÍA TRAS LAS TRANSFORMACIONES ---")
skew_nuevo = var_num_log[variables_sesgadas].skew()
for col in variables_sesgadas:
    print(f"- {col} (Nuevo Skew: {skew_nuevo[col]:.2f})")

--- VARIABLES DETECTADAS PARA TRANSFORMACIÓN ---
- tiempo_seg_cond1 (Skew original: -10.17)
- tiempo_seg_cond2 (Skew original: -7.17)
- tiempo_seg_cond3 (Skew original: -8.28)
- imc (Skew original: 1.14)
- medidas_validas_elasto (Skew original: 5.06)
- intentos_totales_elasto (Skew original: 1.97)
- rigidez_mediana_kpa (Skew original: 7.97)
- rigidez_iqr (Skew original: 18.50)
- ratio_iqr_mediana (Skew original: 18.59)
- cap_iqr (Skew original: 1.84)
- albumina_orina_mg_l (Skew original: 11.53)
- creatinina_orina_umol_l (Skew original: 1.22)
- ratio_albumina_creatinina (Skew original: 14.22)
- peso_flebotomia (Skew original: 1.83)
- proteina_c_reactiva_mg_l (Skew original: 6.95)
- leucocitos_totales (Skew original: 1.14)
- linfocitos_abs (Skew original: 1.16)
- monocitos_abs (Skew original: 1.21)
- neutrofilos_abs (Skew original: 1.39)
- eosinofilos_abs (Skew original: 5.42)
- ancho_distribucion_eritrocitos (Skew original: 5.21)
- trigliceridos_mmol_l (Skew original: 6.92)
- hdl_mmol_l

In [34]:

scaler = StandardScaler()
datos_escalados_array = scaler.fit_transform(var_num_log)

In [35]:
var_num_log_scaled = pd.DataFrame(
    datos_escalados_array,
    columns=var_num_log.columns, # Recuperamos los nombres de las columnas
    index=var_num_log.index      # Recuperamos los índices originales
)
var_num_log_scaled.head(10)

,edad_an,tamano_hogar,psu,ratio_pobreza,tiempo_seg_cond1,tiempo_seg_cond2,tiempo_seg_cond3,tiempo_seg_cond4,pulso,peso_kg,...,plaquetas_totales,volumen_plaquetario_medio,trigliceridos_mmol_l,ldl_martin_mmol_l,hdl_mmol_l,plomo_sangre_umol_l,cadmio_sangre_nmol_l,mercurio_sangre_nmol_l,selenio_sangre_umol_l,manganeso_sangre_nmol_l
0,-0.188751,1.575278,1.021598,0.984181,0.11711,0.191573,0.132493,0.810059,-0.438261,1.282766,...,-0.148788,-0.064880,-0.217923,0.564297,0.991320,-1.128475,-0.198237,0.910452,0.358438,0.143478
1,0.575265,0.208042,1.021598,-0.578577,0.11711,0.191573,0.132493,0.810059,0.158633,1.859173,...,1.324984,-1.395746,2.836066,-0.674377,-1.076357,-0.913991,0.978343,-1.202643,-0.554631,-2.931749
2,-0.049839,0.208042,1.021598,1.226568,0.11711,0.191573,0.132493,0.810059,-0.352991,0.254454,...,-1.146655,0.933270,-0.629579,0.004270,0.050606,-0.913991,0.038614,-0.144919,0.328263,1.263090
3,-0.605487,1.575278,-0.978858,0.021012,0.11711,0.191573,0.132493,0.810059,0.840797,-0.086779,...,-0.440472,0.489648,-0.113511,1.921592,0.050606,0.263839,-0.290164,1.259538,0.298000,-0.321799
4,1.200370,0.208042,1.021598,0.250642,0.11711,0.191573,0.132493,-1.268443,2.716750,-0.294286,...,-0.302306,-1.395746,1.260745,-1.470505,0.742922,2.202996,3.421947,3.363510,3.024857,0.752001
5,-1.508416,-0.475576,-0.978858,1.226568,0.11711,0.191573,0.132493,0.810059,-0.182450,-1.520881,...,-0.102733,1.376892,-1.218732,-1.323370,1.641699,16.169732,0.345159,-0.976735,0.972513,1.507047
6,1.408738,-1.159194,-0.978858,-0.412733,0.11711,0.191573,0.132493,0.810059,-1.802591,0.120728,...,-0.271603,0.156931,-0.670505,-0.762202,0.537385,0.318915,0.906603,-0.011673,-0.488996,0.709928
7,-1.091680,1.575278,1.021598,-1.292981,0.11711,0.191573,0.132493,0.810059,-1.120426,0.019280,...,0.588098,-0.619407,0.523314,0.291392,-0.394170,0.091336,-0.454188,-0.947824,0.328263,-0.478391
8,0.366897,0.891660,-0.978858,1.226568,0.11711,0.191573,0.132493,0.054240,0.158633,0.480406,...,-0.087381,-0.952124,2.461369,-0.142865,-1.142999,-0.159167,-0.518173,-0.024319,-0.521761,-1.554410
9,-0.605487,-1.159194,-0.978858,1.226568,0.11711,0.191573,0.132493,-1.835307,-1.717320,-0.796913,...,-0.732157,-0.064880,-1.170429,-1.027958,-0.265184,0.603880,-0.814587,1.384638,1.224884,-2.172327


In [36]:
var_num_log_scaled.shape


(1919, 46)

In [37]:
# 1. Seleccionar las variables de interés del nuevo dataframe
columnas = ['tiempo_seg_cond4', 'ancho_distribucion_eritrocitos']
X = var_num_log_scaled[columnas].copy()

kmeans = KMeans(n_clusters=3, random_state=42, n_init='auto')
kmeans.fit(X)

# 3. Predecir a qué clúster pertenece cada observación
labels = kmeans.predict(X)

# 4. Guardar las etiquetas temporalmente para el conteo y la gráfica
X['cluster'] = labels

# Mostrar el tamaño de los clústeres
conteos = X['cluster'].value_counts().sort_index()
print("--- Número de filas por clúster (K-Means) ---")
for cluster_id, cantidad in conteos.items():
    print(f"Clúster {cluster_id}: {cantidad} filas")
print("---------------------------------------------\n")

# 5. Calcular el Silhouette Score (¡Excluyendo la etiqueta para no inflarlo!)
score = silhouette_score(X.drop(columns=['cluster']), labels)
print(f"El Silhouette Score para K-Means con k=2 es: {score:.4f}\n")

# 6. Preparar para la visualización (evitar degradado de color)
X['cluster'] = X['cluster'].astype(str)


--- Número de filas por clúster (K-Means) ---
Clúster 0: 1123 filas
Clúster 1: 609 filas
Clúster 2: 187 filas
---------------------------------------------

El Silhouette Score para K-Means con k=2 es: 0.5574



In [38]:
df_train_target = pd.read_csv("../trainset_target.csv")


In [39]:
df_train_target.shape

(1919, 1)

In [40]:
df_train_target1=df_train_target.copy()

In [41]:
df_train_target1['clusters'] = labels
print(df_train_target1.head())

# (Opcional) Ver la relación rápida entre los clústeres y la hipertensión


   hipertension  clusters
0             1         0
1             0         2
2             1         0
3             1         0
4             1         1


In [42]:
print("\n--- Relación: Clúster vs Hipertensión ---")
print(pd.crosstab(df_train_target1['clusters'], df_train_target1['hipertension']))


--- Relación: Clúster vs Hipertensión ---
hipertension    0    1
clusters              
0             658  465
1             297  312
2              94   93


### K-means 'tiempo_seg_cond4', 'eosinofilos_abs' K=4

In [43]:
# 1. Seleccionar las variables de interés del nuevo dataframe
columnas = ['tiempo_seg_cond4', 'eosinofilos_abs']
X2 = var_num_log_scaled[columnas].copy()

kmeans = KMeans(n_clusters=4, random_state=42, n_init='auto')
kmeans.fit(X2)

# 3. Predecir a qué clúster pertenece cada observación
labels2 = kmeans.predict(X2)

# 4. Guardar las etiquetas temporalmente para el conteo y la gráfica
X2['cluster'] = labels2

# Mostrar el tamaño de los clústeres
conteos = X2['cluster'].value_counts().sort_index()
print("--- Número de filas por clúster (K-Means) ---")
for cluster_id, cantidad in conteos.items():
    print(f"Clúster {cluster_id}: {cantidad} filas")
print("---------------------------------------------\n")

# 5. Calcular el Silhouette Score (¡Excluyendo la etiqueta para no inflarlo!)
score = silhouette_score(X2.drop(columns=['cluster']), labels2)
print(f"El Silhouette Score para K-Means con k=2 es: {score:.4f}\n")

# 6. Preparar para la visualización (evitar degradado de color)
X2['cluster'] = X2['cluster'].astype(str)


--- Número de filas por clúster (K-Means) ---
Clúster 0: 563 filas
Clúster 1: 577 filas
Clúster 2: 617 filas
Clúster 3: 162 filas
---------------------------------------------

El Silhouette Score para K-Means con k=2 es: 0.5375



In [44]:
df_train_target2=df_train_target.copy()

In [45]:
df_train_target2['clusters'] = labels2
print(df_train_target2.head())

# (Opcional) Ver la relación rápida entre los clústeres y la hipertensión


   hipertension  clusters
0             1         3
1             0         0
2             1         2
3             1         2
4             1         1


In [46]:
print("\n--- Relación: Clúster vs Hipertensión ---")
print(pd.crosstab(df_train_target2['clusters'], df_train_target2['hipertension']))


--- Relación: Clúster vs Hipertensión ---
hipertension    0    1
clusters              
0             308  255
1             287  290
2             375  242
3              79   83


### clustering jerarquico 'tiempo_seg_cond4', 'plomo_sangre_umol_l' k=3

In [47]:
columnas = ['tiempo_seg_cond4', 'plomo_sangre_umol_l']
X3 = var_num_log_scaled[columnas].copy()

# 2. Inicializar y entrenar el Clustering Jerárquico (k=3)
# Usamos 'ward' por defecto, que minimiza la varianza dentro de los clústeres (ideal para datos escalados)
hc = AgglomerativeClustering(n_clusters=3, metric='euclidean', linkage='ward')

# 3. Ajustar el modelo y predecir las etiquetas en un solo paso
labels3 = hc.fit_predict(X3)

# 4. Guardar las etiquetas temporalmente
X3['cluster'] = labels3

# Mostrar el tamaño de los clústeres
conteos = X3['cluster'].value_counts().sort_index()
print("--- Número de filas por clúster (Jerárquico) ---")
for cluster_id, cantidad in conteos.items():
    print(f"Clúster {cluster_id}: {cantidad} filas")
print("------------------------------------------------\n")

# 5. Calcular el Silhouette Score excluyendo la columna 'cluster'
score = silhouette_score(X3.drop(columns=['cluster']), labels3)
print(f"El Silhouette Score para Clustering Jerárquico con k=3 es: {score:.4f}\n")

# 6. Preparar para Plotly (convertir a string para colores discretos)
X3['cluster'] = X3['cluster'].astype(str)

--- Número de filas por clúster (Jerárquico) ---
Clúster 0: 1397 filas
Clúster 1: 509 filas
Clúster 2: 13 filas
------------------------------------------------

El Silhouette Score para Clustering Jerárquico con k=3 es: 0.5304



In [48]:
df_train_target3=df_train_target.copy()

In [49]:
df_train_target3['clusters'] = labels3
print(df_train_target3.head())

# (Opcional) Ver la relación rápida entre los clústeres y la hipertensión


   hipertension  clusters
0             1         0
1             0         0
2             1         0
3             1         0
4             1         1


In [50]:
print("\n--- Relación: Clúster vs Hipertensión ---")
print(pd.crosstab(df_train_target3['clusters'], df_train_target3['hipertension']))


--- Relación: Clúster vs Hipertensión ---
hipertension    0    1
clusters              
0             788  609
1             257  252
2               4    9


### clustering jerarquico 'tiempo_seg_cond4', 'eosinofilos_abs' k=4

In [51]:
columnas = ['tiempo_seg_cond4', 'eosinofilos_abs']
X4 = var_num_log_scaled[columnas].copy()

# 2. Inicializar y entrenar el Clustering Jerárquico (k=4)
# Usamos 'ward' por defecto, que minimiza la varianza dentro de los clústeres (ideal para datos escalados)
hc = AgglomerativeClustering(n_clusters=4, metric='euclidean', linkage='ward')

# 3. Ajustar el modelo y predecir las etiquetas en un solo paso
labels4 = hc.fit_predict(X4)

# 4. Guardar las etiquetas temporalmente
X4['cluster'] = labels4

# Mostrar el tamaño de los clústeres
conteos = X4['cluster'].value_counts().sort_index()
print("--- Número de filas por clúster (Jerárquico) ---")
for cluster_id, cantidad in conteos.items():
    print(f"Clúster {cluster_id}: {cantidad} filas")
print("------------------------------------------------\n")

# 5. Calcular el Silhouette Score excluyendo la columna 'cluster'
score = silhouette_score(X4.drop(columns=['cluster']), labels4)
print(f"El Silhouette Score para Clustering Jerárquico con k=3 es: {score:.4f}\n")

# 6. Preparar para Plotly (convertir a string para colores discretos)
X4['cluster'] = X4['cluster'].astype(str)

--- Número de filas por clúster (Jerárquico) ---
Clúster 0: 1140 filas
Clúster 1: 458 filas
Clúster 2: 291 filas
Clúster 3: 30 filas
------------------------------------------------

El Silhouette Score para Clustering Jerárquico con k=3 es: 0.5056



In [52]:
df_train_target4=df_train_target.copy()

In [53]:
df_train_target4['clusters'] = labels4
print(df_train_target4.head())

# (Opcional) Ver la relación rápida entre los clústeres y la hipertensión


   hipertension  clusters
0             1         2
1             0         0
2             1         0
3             1         0
4             1         1


In [54]:
print("\n--- Relación: Clúster vs Hipertensión ---")
print(pd.crosstab(df_train_target4['clusters'], df_train_target4['hipertension']))


--- Relación: Clúster vs Hipertensión ---
hipertension    0    1
clusters              
0             667  473
1             221  237
2             147  144
3              14   16


----------------------------------------------------------------------------------------------

### Analisi clusters variables codigo genertico. 

#### K-Means 'hemoglobina_g_dl', 'ancho_distribucion_eritrocitos' k=2

In [55]:
# 1. Seleccionar las variables de interés del nuevo dataframe
columnas = ['hemoglobina_g_dl', 'ancho_distribucion_eritrocitos']
X = var_num_log_scaled[columnas].copy()

kmeans = KMeans(n_clusters=2, random_state=42, n_init='auto')
kmeans.fit(X)

# 3. Predecir a qué clúster pertenece cada observación
labels = kmeans.predict(X)

# 4. Guardar las etiquetas temporalmente para el conteo y la gráfica
X['cluster'] = labels

# Mostrar el tamaño de los clústeres
conteos = X['cluster'].value_counts().sort_index()
print("--- Número de filas por clúster (K-Means) ---")
for cluster_id, cantidad in conteos.items():
    print(f"Clúster {cluster_id}: {cantidad} filas")
print("---------------------------------------------\n")

# 5. Calcular el Silhouette Score (¡Excluyendo la etiqueta para no inflarlo!)
score = silhouette_score(X.drop(columns=['cluster']), labels)
print(f"El Silhouette Score para K-Means con k=2 es: {score:.4f}\n")

# 6. Preparar para la visualización (evitar degradado de color)
X['cluster'] = X['cluster'].astype(str)


--- Número de filas por clúster (K-Means) ---
Clúster 0: 1590 filas
Clúster 1: 329 filas
---------------------------------------------

El Silhouette Score para K-Means con k=2 es: 0.4765



In [56]:
df_train_target1=df_train_target.copy()

In [57]:
df_train_target1['clusters'] = labels
print(df_train_target1.head())

# (Opcional) Ver la relación rápida entre los clústeres y la hipertensión


   hipertension  clusters
0             1         0
1             0         1
2             1         0
3             1         0
4             1         1


In [58]:
print("\n--- Relación: Clúster vs Hipertensión ---")
print(pd.crosstab(df_train_target1['clusters'], df_train_target1['hipertension']))


--- Relación: Clúster vs Hipertensión ---
hipertension    0    1
clusters              
0             881  709
1             168  161


#### K-Means 'ancho_distribucion_eritrocitos', 'edad_an' k=3

In [59]:
# 1. Seleccionar las variables de interés del nuevo dataframe
columnas = ['ancho_distribucion_eritrocitos', 'edad_an']
X = var_num_log_scaled[columnas].copy()

kmeans = KMeans(n_clusters=3, random_state=42, n_init='auto')
kmeans.fit(X)

# 3. Predecir a qué clúster pertenece cada observación
labels = kmeans.predict(X)

# 4. Guardar las etiquetas temporalmente para el conteo y la gráfica
X['cluster'] = labels

# Mostrar el tamaño de los clústeres
conteos = X['cluster'].value_counts().sort_index()
print("--- Número de filas por clúster (K-Means) ---")
for cluster_id, cantidad in conteos.items():
    print(f"Clúster {cluster_id}: {cantidad} filas")
print("---------------------------------------------\n")

# 5. Calcular el Silhouette Score (¡Excluyendo la etiqueta para no inflarlo!)
score = silhouette_score(X.drop(columns=['cluster']), labels)
print(f"El Silhouette Score para K-Means con k=3 es: {score:.4f}\n")

# 6. Preparar para la visualización (evitar degradado de color)
X['cluster'] = X['cluster'].astype(str)


--- Número de filas por clúster (K-Means) ---
Clúster 0: 792 filas
Clúster 1: 930 filas
Clúster 2: 197 filas
---------------------------------------------

El Silhouette Score para K-Means con k=3 es: 0.4788



In [60]:
df_train_target1=df_train_target.copy()

In [61]:
df_train_target1['clusters'] = labels
print(df_train_target1.head())

# (Opcional) Ver la relación rápida entre los clústeres y la hipertensión


   hipertension  clusters
0             1         0
1             0         2
2             1         0
3             1         0
4             1         2


In [62]:
print("\n--- Relación: Clúster vs Hipertensión ---")
print(pd.crosstab(df_train_target1['clusters'], df_train_target1['hipertension']))


--- Relación: Clúster vs Hipertensión ---
hipertension    0    1
clusters              
0             525  267
1             436  494
2              88  109


#### K-Means 'ancho_distribucion_eritrocitos', 'edad_an' k=4

In [63]:
# 1. Seleccionar las variables de interés del nuevo dataframe
columnas = ['ancho_distribucion_eritrocitos', 'edad_an']
X = var_num_log_scaled[columnas].copy()

kmeans = KMeans(n_clusters=4, random_state=42, n_init='auto')
kmeans.fit(X)

# 3. Predecir a qué clúster pertenece cada observación
labels = kmeans.predict(X)

# 4. Guardar las etiquetas temporalmente para el conteo y la gráfica
X['cluster'] = labels

# Mostrar el tamaño de los clústeres
conteos = X['cluster'].value_counts().sort_index()
print("--- Número de filas por clúster (K-Means) ---")
for cluster_id, cantidad in conteos.items():
    print(f"Clúster {cluster_id}: {cantidad} filas")
print("---------------------------------------------\n")

# 5. Calcular el Silhouette Score (¡Excluyendo la etiqueta para no inflarlo!)
score = silhouette_score(X.drop(columns=['cluster']), labels)
print(f"El Silhouette Score para K-Means con k=4 es: {score:.4f}\n")

# 6. Preparar para la visualización (evitar degradado de color)
X['cluster'] = X['cluster'].astype(str)


--- Número de filas por clúster (K-Means) ---
Clúster 0: 767 filas
Clúster 1: 798 filas
Clúster 2: 307 filas
Clúster 3: 47 filas
---------------------------------------------

El Silhouette Score para K-Means con k=4 es: 0.4449



In [64]:
df_train_target1=df_train_target.copy()

In [65]:
df_train_target1['clusters'] = labels
print(df_train_target1.head())

# (Opcional) Ver la relación rápida entre los clústeres y la hipertensión


   hipertension  clusters
0             1         2
1             0         2
2             1         1
3             1         0
4             1         2


In [66]:
print("\n--- Relación: Clúster vs Hipertensión ---")
print(pd.crosstab(df_train_target1['clusters'], df_train_target1['hipertension']))


--- Relación: Clúster vs Hipertensión ---
hipertension    0    1
clusters              
0             516  251
1             376  422
2             127  180
3              30   17


#### K-Means 'tamano_hogar', 'edad_an' k=5

In [67]:
# 1. Seleccionar las variables de interés del nuevo dataframe
columnas = ['tamano_hogar', 'edad_an']
X = var_num_log_scaled[columnas].copy()

kmeans = KMeans(n_clusters=5, random_state=42, n_init='auto')
kmeans.fit(X)

# 3. Predecir a qué clúster pertenece cada observación
labels = kmeans.predict(X)

# 4. Guardar las etiquetas temporalmente para el conteo y la gráfica
X['cluster'] = labels

# Mostrar el tamaño de los clústeres
conteos = X['cluster'].value_counts().sort_index()
print("--- Número de filas por clúster (K-Means) ---")
for cluster_id, cantidad in conteos.items():
    print(f"Clúster {cluster_id}: {cantidad} filas")
print("---------------------------------------------\n")

# 5. Calcular el Silhouette Score (¡Excluyendo la etiqueta para no inflarlo!)
score = silhouette_score(X.drop(columns=['cluster']), labels)
print(f"El Silhouette Score para K-Means con k=5 es: {score:.4f}\n")

# 6. Preparar para la visualización (evitar degradado de color)
X['cluster'] = X['cluster'].astype(str)


--- Número de filas por clúster (K-Means) ---
Clúster 0: 362 filas
Clúster 1: 716 filas
Clúster 2: 348 filas
Clúster 3: 190 filas
Clúster 4: 303 filas
---------------------------------------------

El Silhouette Score para K-Means con k=5 es: 0.4426



In [68]:
df_train_target1=df_train_target.copy()

In [69]:
df_train_target1['clusters'] = labels
print(df_train_target1.head())

# (Opcional) Ver la relación rápida entre los clústeres y la hipertensión


   hipertension  clusters
0             1         3
1             0         4
2             1         4
3             1         3
4             1         4


In [70]:
print("\n--- Relación: Clúster vs Hipertensión ---")
print(pd.crosstab(df_train_target1['clusters'], df_train_target1['hipertension']))


--- Relación: Clúster vs Hipertensión ---
hipertension    0    1
clusters              
0             251  111
1             336  380
2             231  117
3             101   89
4             130  173


### clustering jerarquico 'tamano_hogar', 'edad_an' k=4

In [71]:
columnas = ['tamano_hogar', 'edad_an']
X3 = var_num_log_scaled[columnas].copy()

# 2. Inicializar y entrenar el Clustering Jerárquico (k=4)
# Usamos 'ward' por defecto, que minimiza la varianza dentro de los clústeres (ideal para datos escalados)
hc = AgglomerativeClustering(n_clusters=4, metric='euclidean', linkage='ward')

# 3. Ajustar el modelo y predecir las etiquetas en un solo paso
labels3 = hc.fit_predict(X3)

# 4. Guardar las etiquetas temporalmente
X3['cluster'] = labels3

# Mostrar el tamaño de los clústeres
conteos = X3['cluster'].value_counts().sort_index()
print("--- Número de filas por clúster (Jerárquico) ---")
for cluster_id, cantidad in conteos.items():
    print(f"Clúster {cluster_id}: {cantidad} filas")
print("------------------------------------------------\n")

# 5. Calcular el Silhouette Score excluyendo la columna 'cluster'
score = silhouette_score(X3.drop(columns=['cluster']), labels3)
print(f"El Silhouette Score para Clustering Jerárquico con k=4 es: {score:.4f}\n")

# 6. Preparar para Plotly (convertir a string para colores discretos)
X3['cluster'] = X3['cluster'].astype(str)

--- Número de filas por clúster (Jerárquico) ---
Clúster 0: 177 filas
Clúster 1: 457 filas
Clúster 2: 711 filas
Clúster 3: 574 filas
------------------------------------------------

El Silhouette Score para Clustering Jerárquico con k=4 es: 0.4154



In [72]:
df_train_target3=df_train_target.copy()

In [73]:
df_train_target3['clusters'] = labels3
print(df_train_target3.head())

# (Opcional) Ver la relación rápida entre los clústeres y la hipertensión


   hipertension  clusters
0             1         3
1             0         2
2             1         3
3             1         3
4             1         2


In [74]:
print("\n--- Relación: Clúster vs Hipertensión ---")
print(pd.crosstab(df_train_target3['clusters'], df_train_target3['hipertension']))


--- Relación: Clúster vs Hipertensión ---
hipertension    0    1
clusters              
0              77  100
1             278  179
2             329  382
3             365  209


### clustering jerarquico 'tamano_hogar', 'edad_an' k=5

In [75]:
columnas = ['tamano_hogar', 'edad_an']
X3 = var_num_log_scaled[columnas].copy()

# 2. Inicializar y entrenar el Clustering Jerárquico (k=4)
# Usamos 'ward' por defecto, que minimiza la varianza dentro de los clústeres (ideal para datos escalados)
hc = AgglomerativeClustering(n_clusters=5, metric='euclidean', linkage='ward')

# 3. Ajustar el modelo y predecir las etiquetas en un solo paso
labels3 = hc.fit_predict(X3)

# 4. Guardar las etiquetas temporalmente
X3['cluster'] = labels3

# Mostrar el tamaño de los clústeres
conteos = X3['cluster'].value_counts().sort_index()
print("--- Número de filas por clúster (Jerárquico) ---")
for cluster_id, cantidad in conteos.items():
    print(f"Clúster {cluster_id}: {cantidad} filas")
print("------------------------------------------------\n")

# 5. Calcular el Silhouette Score excluyendo la columna 'cluster'
score = silhouette_score(X3.drop(columns=['cluster']), labels3)
print(f"El Silhouette Score para Clustering Jerárquico con k=4 es: {score:.4f}\n")

# 6. Preparar para Plotly (convertir a string para colores discretos)
X3['cluster'] = X3['cluster'].astype(str)

--- Número de filas por clúster (Jerárquico) ---
Clúster 0: 457 filas
Clúster 1: 574 filas
Clúster 2: 711 filas
Clúster 3: 104 filas
Clúster 4: 73 filas
------------------------------------------------

El Silhouette Score para Clustering Jerárquico con k=4 es: 0.4150



In [76]:
df_train_target3=df_train_target.copy()

In [77]:
df_train_target3['clusters'] = labels3
print(df_train_target3.head())

# (Opcional) Ver la relación rápida entre los clústeres y la hipertensión


   hipertension  clusters
0             1         1
1             0         2
2             1         1
3             1         1
4             1         2


In [78]:
print("\n--- Relación: Clúster vs Hipertensión ---")
print(pd.crosstab(df_train_target3['clusters'], df_train_target3['hipertension']))


--- Relación: Clúster vs Hipertensión ---
hipertension    0    1
clusters              
0             278  179
1             365  209
2             329  382
3              41   63
4              36   37


### clustering gmm 'ancho_distribucion_eritrocitos', 'edad_an' k=3

In [79]:
columnas = ['ancho_distribucion_eritrocitos', 'edad_an']
X3 = var_num_log_scaled[columnas].copy()

# 2. Inicializar y entrenar el modelo GMM (k=3)
# Utilizamos n_components en lugar de n_clusters, y fijamos la semilla aleatoria
gmm = GaussianMixture(n_components=3, random_state=42)

# 3. Ajustar el modelo y predecir las etiquetas
# GMM también soporta fit_predict, así que podemos mantenerlo en un solo paso
labels3 = gmm.fit_predict(X3)

# 4. Guardar las etiquetas temporalmente
X3['cluster'] = labels3

# Mostrar el tamaño de los clústeres
conteos = X3['cluster'].value_counts().sort_index()
print("--- Número de filas por clúster (GMM) ---")
for cluster_id, cantidad in conteos.items():
    print(f"Clúster {cluster_id}: {cantidad} filas")
print("-----------------------------------------\n")

# 5. Calcular el Silhouette Score excluyendo la columna 'cluster'
score = silhouette_score(X3.drop(columns=['cluster']), labels3)
print(f"El Silhouette Score para GMM con k=4 es: {score:.4f}\n")

# 6. Preparar para Plotly (convertir a string para colores discretos)
X3['cluster'] = X3['cluster'].astype(str)

--- Número de filas por clúster (GMM) ---
Clúster 0: 915 filas
Clúster 1: 844 filas
Clúster 2: 160 filas
-----------------------------------------

El Silhouette Score para GMM con k=4 es: 0.4592



In [80]:
df_train_target3=df_train_target.copy()

In [81]:
df_train_target3['clusters'] = labels3
print(df_train_target3.head())

# (Opcional) Ver la relación rápida entre los clústeres y la hipertensión


   hipertension  clusters
0             1         0
1             0         1
2             1         0
3             1         0
4             1         1


In [82]:
print("\n--- Relación: Clúster vs Hipertensión ---")
print(pd.crosstab(df_train_target3['clusters'], df_train_target3['hipertension']))


--- Relación: Clúster vs Hipertensión ---
hipertension    0    1
clusters              
0             580  335
1             385  459
2              84   76


### clustering gmm 'ancho_distribucion_eritrocitos', 'edad_an' k=4

In [83]:
columnas = ['ancho_distribucion_eritrocitos', 'edad_an']
X3 = var_num_log_scaled[columnas].copy()

# 2. Inicializar y entrenar el modelo GMM (k=4)
# Utilizamos n_components en lugar de n_clusters, y fijamos la semilla aleatoria
gmm = GaussianMixture(n_components=4, random_state=42)

# 3. Ajustar el modelo y predecir las etiquetas
# GMM también soporta fit_predict, así que podemos mantenerlo en un solo paso
labels3 = gmm.fit_predict(X3)

# 4. Guardar las etiquetas temporalmente
X3['cluster'] = labels3

# Mostrar el tamaño de los clústeres
conteos = X3['cluster'].value_counts().sort_index()
print("--- Número de filas por clúster (GMM) ---")
for cluster_id, cantidad in conteos.items():
    print(f"Clúster {cluster_id}: {cantidad} filas")
print("-----------------------------------------\n")

# 5. Calcular el Silhouette Score excluyendo la columna 'cluster'
score = silhouette_score(X3.drop(columns=['cluster']), labels3)
print(f"El Silhouette Score para GMM con k=4 es: {score:.4f}\n")

# 6. Preparar para Plotly (convertir a string para colores discretos)
X3['cluster'] = X3['cluster'].astype(str)

--- Número de filas por clúster (GMM) ---
Clúster 0: 904 filas
Clúster 1: 765 filas
Clúster 2: 192 filas
Clúster 3: 58 filas
-----------------------------------------

El Silhouette Score para GMM con k=4 es: 0.4360



In [84]:
df_train_target3=df_train_target.copy()

In [85]:
df_train_target3['clusters'] = labels3
print(df_train_target3.head())

# (Opcional) Ver la relación rápida entre los clústeres y la hipertensión


   hipertension  clusters
0             1         0
1             0         2
2             1         0
3             1         0
4             1         2


In [86]:
print("\n--- Relación: Clúster vs Hipertensión ---")
print(pd.crosstab(df_train_target3['clusters'], df_train_target3['hipertension']))


--- Relación: Clúster vs Hipertensión ---
hipertension    0    1
clusters              
0             575  329
1             354  411
2              81  111
3              39   19


### clustering gmm 'ancho_distribucion_eritrocitos', 'tamano_hogar' k=5

In [87]:
columnas = ['ancho_distribucion_eritrocitos', 'tamano_hogar']
X3 = var_num_log_scaled[columnas].copy()

# 2. Inicializar y entrenar el modelo GMM (k=3)
# Utilizamos n_components en lugar de n_clusters, y fijamos la semilla aleatoria
gmm = GaussianMixture(n_components=5, random_state=42)

# 3. Ajustar el modelo y predecir las etiquetas
# GMM también soporta fit_predict, así que podemos mantenerlo en un solo paso
labels3 = gmm.fit_predict(X3)

# 4. Guardar las etiquetas temporalmente
X3['cluster'] = labels3

# Mostrar el tamaño de los clústeres
conteos = X3['cluster'].value_counts().sort_index()
print("--- Número de filas por clúster (GMM) ---")
for cluster_id, cantidad in conteos.items():
    print(f"Clúster {cluster_id}: {cantidad} filas")
print("-----------------------------------------\n")

# 5. Calcular el Silhouette Score excluyendo la columna 'cluster'
score = silhouette_score(X3.drop(columns=['cluster']), labels3)
print(f"El Silhouette Score para GMM con k=4 es: {score:.4f}\n")

# 6. Preparar para Plotly (convertir a string para colores discretos)
X3['cluster'] = X3['cluster'].astype(str)

--- Número de filas por clúster (GMM) ---
Clúster 0: 513 filas
Clúster 1: 72 filas
Clúster 2: 226 filas
Clúster 3: 873 filas
Clúster 4: 235 filas
-----------------------------------------

El Silhouette Score para GMM con k=4 es: 0.3919



In [88]:
df_train_target3=df_train_target.copy()

In [89]:
df_train_target3['clusters'] = labels3
print(df_train_target3.head())

# (Opcional) Ver la relación rápida entre los clústeres y la hipertensión


   hipertension  clusters
0             1         2
1             0         4
2             1         0
3             1         2
4             1         4


In [90]:
print("\n--- Relación: Clúster vs Hipertensión ---")
print(pd.crosstab(df_train_target3['clusters'], df_train_target3['hipertension']))


--- Relación: Clúster vs Hipertensión ---
hipertension    0    1
clusters              
0             311  202
1              40   32
2             107  119
3             487  386
4             104  131


-------------------------------------------------------------------

#### Análisis clusters variables codigo genético

In [91]:
trainset = pd.read_csv("../trainset.csv")

In [92]:
trainset.head()

,edad_an,tamano_hogar,psu,ratio_pobreza,tiempo_seg_cond1,tiempo_seg_cond2,tiempo_seg_cond3,tiempo_seg_cond4,pulso,peso_kg,...,tamano_manguito_pa_4.0,tamano_manguito_pa_5.0,estado_elastografia_1,tipo_sonda_elasto_b'M',tipo_sonda_elasto_b'XL',raza_etnia_2.0,raza_etnia_3.0,raza_etnia_4.0,raza_etnia_6.0,raza_etnia_7.0
0,-0.188751,1.575278,1.021598,0.984181,0.144067,0.245103,0.162136,0.810059,-0.438261,1.282766,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
1,0.575265,0.208042,1.021598,-0.578577,0.144067,0.245103,0.162136,0.810059,0.158633,1.859173,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
2,-0.049839,0.208042,1.021598,1.226568,0.144067,0.245103,0.162136,0.810059,-0.352991,0.254454,...,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
3,-0.605487,1.575278,-0.978858,0.021012,0.144067,0.245103,0.162136,0.810059,0.840797,-0.086779,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
4,1.200370,0.208042,1.021598,0.250642,0.144067,0.245103,0.162136,-1.268443,2.716750,-0.294286,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [93]:
from kmodes.kprototypes import KPrototypes

# Tu lista de columnas
columnas_genetico = [
    "estado_examen_balance_1", "rigidez_iqr", "cap_iqr", "intentos_totales_elasto",
    "fracturas_1", "largo_brazo_superior_cm", "mercurio_sangre_nmol_l", 
    "periodo_examen_1", "raza_etnia_4.0", "tiempo_seg_cond3", "hdl_mmol_l", 
    "tipo_sonda_elasto_b'M'", "raza_etnia_6.0", "medicacion_osteoporosis_1", 
    "creatinina_orina_umol_l", "hemoglobina_g_dl", "tiempo_seg_cond1", 
    "elegibilidad_balance_1", "convulsiones_1", "peso_kg", "psu", 
    "volumen_plaquetario_medio", "ancho_distribucion_eritrocitos", 
    "medidas_validas_elasto", "conc_hemoglobina_media", "tamano_hogar", 
    "pulso", "plaquetas_totales", "edad_an", "trigliceridos_mmol_l", "raza_etnia_2.0"
]

# 1. Crear el dataframe de trabajo
X = trainset[columnas_genetico].copy()

# 2. Separar dinámicamente variables Categóricas y Numéricas
# Consideramos categóricas aquellas tipo 'object' (texto) o con 2 o menos valores únicos (binarias 0/1)
cat_cols = [col for col in X.columns if X[col].dtype == 'object' or X[col].nunique() <= 2]
num_cols = [col for col in X.columns if col not in cat_cols]

print(f"Variables categóricas detectadas ({len(cat_cols)}):", cat_cols)
print(f"Variables numéricas detectadas ({len(num_cols)}):", num_cols)

# 4. Obtener los índices de las columnas categóricas
# El algoritmo exige que le digamos en qué posición exacta de la tabla están las categorías
cat_idx = [X.columns.get_loc(col) for col in cat_cols]

# 5. Inicializar y entrenar K-Prototypes (k=2)
# 'Cao' es el mejor método de inicialización para K-Prototypes
kproto = KPrototypes(n_clusters=2, init='Cao', random_state=42, n_jobs=-1)

# A diferencia de scikit-learn, aquí debemos pasar los valores como matriz numpy (.values)
# y especificar qué índices son categóricos. (Esto puede tardar un poco dependiendo del tamaño del dataset)
print("\nEntrenando K-Prototypes... (esto puede tardar unos segundos/minutos)")
labels = kproto.fit_predict(X.values, categorical=cat_idx)

# 6. Guardar los resultados y mostrar tamaños
X['cluster'] = labels
conteos = X['cluster'].value_counts().sort_index()

print("\n--- Número de filas por clúster (K-Prototypes) ---")
for cluster_id, cantidad in conteos.items():
    print(f"Clúster {cluster_id}: {cantidad} filas")
print("--------------------------------------------------")

# La métrica nativa de evaluación de K-Prototypes es el "costo" (menor es mejor)
print(f"Costo del modelo: {kproto.cost_:.2f}")

Variables categóricas detectadas (11): ['estado_examen_balance_1', 'fracturas_1', 'periodo_examen_1', 'raza_etnia_4.0', "tipo_sonda_elasto_b'M'", 'raza_etnia_6.0', 'medicacion_osteoporosis_1', 'elegibilidad_balance_1', 'convulsiones_1', 'psu', 'raza_etnia_2.0']
Variables numéricas detectadas (20): ['rigidez_iqr', 'cap_iqr', 'intentos_totales_elasto', 'largo_brazo_superior_cm', 'mercurio_sangre_nmol_l', 'tiempo_seg_cond3', 'hdl_mmol_l', 'creatinina_orina_umol_l', 'hemoglobina_g_dl', 'tiempo_seg_cond1', 'peso_kg', 'volumen_plaquetario_medio', 'ancho_distribucion_eritrocitos', 'medidas_validas_elasto', 'conc_hemoglobina_media', 'tamano_hogar', 'pulso', 'plaquetas_totales', 'edad_an', 'trigliceridos_mmol_l']

Entrenando K-Prototypes... (esto puede tardar unos segundos/minutos)

--- Número de filas por clúster (K-Prototypes) ---
Clúster 0: 936 filas
Clúster 1: 983 filas
--------------------------------------------------
Costo del modelo: 37376.14


In [94]:
import gower
from sklearn.metrics import silhouette_score

# 1. Asegurarnos de que las categóricas sean texto para que Gower las evalúe bien
X_eval = X.copy()
for col in cat_cols:
    X_eval[col] = X_eval[col].astype(str)

# 2. Calcular la matriz de distancias (El árbitro imparcial)
print("\nCalculando matriz de Gower para evaluación...")
dist_matrix_eval = gower.gower_matrix(X_eval)

# 3. Calcular el Silhouette Score usando las etiquetas que generó K-Prototypes
score_kproto = silhouette_score(dist_matrix_eval, labels, metric='precomputed')

print(f"Silhouette Score (Gower) de K-Prototypes para k=4: {score_kproto:.4f}")


Calculando matriz de Gower para evaluación...
Silhouette Score (Gower) de K-Prototypes para k=4: 0.2075


In [95]:
df_train_target=df_train_target.copy()

In [96]:
df_train_target['clusters'] = labels
print(df_train_target.head())

# (Opcional) Ver la relación rápida entre los clústeres y la hipertensión


   hipertension  clusters
0             1         1
1             0         1
2             1         0
3             1         0
4             1         1


In [97]:
print("\n--- Relación: Clúster vs Hipertensión ---")
print(pd.crosstab(df_train_target['clusters'], df_train_target['hipertension']))


--- Relación: Clúster vs Hipertensión ---
hipertension    0    1
clusters              
0             450  486
1             599  384


K=3

In [98]:
from kmodes.kprototypes import KPrototypes

# Tu lista de columnas
columnas_genetico = [
    "estado_examen_balance_1", "rigidez_iqr", "cap_iqr", "intentos_totales_elasto",
    "fracturas_1", "largo_brazo_superior_cm", "mercurio_sangre_nmol_l", 
    "periodo_examen_1", "raza_etnia_4.0", "tiempo_seg_cond3", "hdl_mmol_l", 
    "tipo_sonda_elasto_b'M'", "raza_etnia_6.0", "medicacion_osteoporosis_1", 
    "creatinina_orina_umol_l", "hemoglobina_g_dl", "tiempo_seg_cond1", 
    "elegibilidad_balance_1", "convulsiones_1", "peso_kg", "psu", 
    "volumen_plaquetario_medio", "ancho_distribucion_eritrocitos", 
    "medidas_validas_elasto", "conc_hemoglobina_media", "tamano_hogar", 
    "pulso", "plaquetas_totales", "edad_an", "trigliceridos_mmol_l", "raza_etnia_2.0"
]

# 1. Crear el dataframe de trabajo
X = trainset[columnas_genetico].copy()

# 2. Separar dinámicamente variables Categóricas y Numéricas
# Consideramos categóricas aquellas tipo 'object' (texto) o con 2 o menos valores únicos (binarias 0/1)
cat_cols = [col for col in X.columns if X[col].dtype == 'object' or X[col].nunique() <= 2]
num_cols = [col for col in X.columns if col not in cat_cols]

print(f"Variables categóricas detectadas ({len(cat_cols)}):", cat_cols)
print(f"Variables numéricas detectadas ({len(num_cols)}):", num_cols)

# 4. Obtener los índices de las columnas categóricas
# El algoritmo exige que le digamos en qué posición exacta de la tabla están las categorías
cat_idx = [X.columns.get_loc(col) for col in cat_cols]

# 5. Inicializar y entrenar K-Prototypes (k=2)
# 'Cao' es el mejor método de inicialización para K-Prototypes
kproto = KPrototypes(n_clusters=3, init='Cao', random_state=42, n_jobs=-1)

# A diferencia de scikit-learn, aquí debemos pasar los valores como matriz numpy (.values)
# y especificar qué índices son categóricos. (Esto puede tardar un poco dependiendo del tamaño del dataset)
print("\nEntrenando K-Prototypes... (esto puede tardar unos segundos/minutos)")
labels = kproto.fit_predict(X.values, categorical=cat_idx)

# 6. Guardar los resultados y mostrar tamaños
X['cluster'] = labels
conteos = X['cluster'].value_counts().sort_index()

print("\n--- Número de filas por clúster (K-Prototypes) ---")
for cluster_id, cantidad in conteos.items():
    print(f"Clúster {cluster_id}: {cantidad} filas")
print("--------------------------------------------------")

# La métrica nativa de evaluación de K-Prototypes es el "costo" (menor es mejor)
print(f"Costo del modelo: {kproto.cost_:.2f}")

Variables categóricas detectadas (11): ['estado_examen_balance_1', 'fracturas_1', 'periodo_examen_1', 'raza_etnia_4.0', "tipo_sonda_elasto_b'M'", 'raza_etnia_6.0', 'medicacion_osteoporosis_1', 'elegibilidad_balance_1', 'convulsiones_1', 'psu', 'raza_etnia_2.0']
Variables numéricas detectadas (20): ['rigidez_iqr', 'cap_iqr', 'intentos_totales_elasto', 'largo_brazo_superior_cm', 'mercurio_sangre_nmol_l', 'tiempo_seg_cond3', 'hdl_mmol_l', 'creatinina_orina_umol_l', 'hemoglobina_g_dl', 'tiempo_seg_cond1', 'peso_kg', 'volumen_plaquetario_medio', 'ancho_distribucion_eritrocitos', 'medidas_validas_elasto', 'conc_hemoglobina_media', 'tamano_hogar', 'pulso', 'plaquetas_totales', 'edad_an', 'trigliceridos_mmol_l']

Entrenando K-Prototypes... (esto puede tardar unos segundos/minutos)

--- Número de filas por clúster (K-Prototypes) ---
Clúster 0: 342 filas
Clúster 1: 739 filas
Clúster 2: 838 filas
--------------------------------------------------
Costo del modelo: 35665.28


In [99]:
import gower
from sklearn.metrics import silhouette_score

# 1. Asegurarnos de que las categóricas sean texto para que Gower las evalúe bien
X_eval = X.copy()
for col in cat_cols:
    X_eval[col] = X_eval[col].astype(str)

# 2. Calcular la matriz de distancias (El árbitro imparcial)
print("\nCalculando matriz de Gower para evaluación...")
dist_matrix_eval = gower.gower_matrix(X_eval)

# 3. Calcular el Silhouette Score usando las etiquetas que generó K-Prototypes
score_kproto = silhouette_score(dist_matrix_eval, labels, metric='precomputed')

print(f"Silhouette Score (Gower) de K-Prototypes para k=4: {score_kproto:.4f}")


Calculando matriz de Gower para evaluación...
Silhouette Score (Gower) de K-Prototypes para k=4: 0.1318


In [100]:
df_train_target=df_train_target.copy()

In [101]:
df_train_target['clusters'] = labels
print(df_train_target.head())

# (Opcional) Ver la relación rápida entre los clústeres y la hipertensión


   hipertension  clusters
0             1         0
1             0         0
2             1         1
3             1         1
4             1         2


In [102]:
print("\n--- Relación: Clúster vs Hipertensión ---")
print(pd.crosstab(df_train_target['clusters'], df_train_target['hipertension']))


--- Relación: Clúster vs Hipertensión ---
hipertension    0    1
clusters              
0             163  179
1             364  375
2             522  316


K=4

In [103]:
from kmodes.kprototypes import KPrototypes

# Tu lista de columnas
columnas_genetico = [
    "estado_examen_balance_1", "rigidez_iqr", "cap_iqr", "intentos_totales_elasto",
    "fracturas_1", "largo_brazo_superior_cm", "mercurio_sangre_nmol_l", 
    "periodo_examen_1", "raza_etnia_4.0", "tiempo_seg_cond3", "hdl_mmol_l", 
    "tipo_sonda_elasto_b'M'", "raza_etnia_6.0", "medicacion_osteoporosis_1", 
    "creatinina_orina_umol_l", "hemoglobina_g_dl", "tiempo_seg_cond1", 
    "elegibilidad_balance_1", "convulsiones_1", "peso_kg", "psu", 
    "volumen_plaquetario_medio", "ancho_distribucion_eritrocitos", 
    "medidas_validas_elasto", "conc_hemoglobina_media", "tamano_hogar", 
    "pulso", "plaquetas_totales", "edad_an", "trigliceridos_mmol_l", "raza_etnia_2.0"
]

# 1. Crear el dataframe de trabajo
X = trainset[columnas_genetico].copy()

# 2. Separar dinámicamente variables Categóricas y Numéricas
# Consideramos categóricas aquellas tipo 'object' (texto) o con 2 o menos valores únicos (binarias 0/1)
cat_cols = [col for col in X.columns if X[col].dtype == 'object' or X[col].nunique() <= 2]
num_cols = [col for col in X.columns if col not in cat_cols]

print(f"Variables categóricas detectadas ({len(cat_cols)}):", cat_cols)
print(f"Variables numéricas detectadas ({len(num_cols)}):", num_cols)

# 4. Obtener los índices de las columnas categóricas
# El algoritmo exige que le digamos en qué posición exacta de la tabla están las categorías
cat_idx = [X.columns.get_loc(col) for col in cat_cols]

# 5. Inicializar y entrenar K-Prototypes (k=2)
# 'Cao' es el mejor método de inicialización para K-Prototypes
kproto = KPrototypes(n_clusters=4, init='Cao', random_state=42, n_jobs=-1)

# A diferencia de scikit-learn, aquí debemos pasar los valores como matriz numpy (.values)
# y especificar qué índices son categóricos. (Esto puede tardar un poco dependiendo del tamaño del dataset)
print("\nEntrenando K-Prototypes... (esto puede tardar unos segundos/minutos)")
labels = kproto.fit_predict(X.values, categorical=cat_idx)

# 6. Guardar los resultados y mostrar tamaños
X['cluster'] = labels
conteos = X['cluster'].value_counts().sort_index()

print("\n--- Número de filas por clúster (K-Prototypes) ---")
for cluster_id, cantidad in conteos.items():
    print(f"Clúster {cluster_id}: {cantidad} filas")
print("--------------------------------------------------")

# La métrica nativa de evaluación de K-Prototypes es el "costo" (menor es mejor)
print(f"Costo del modelo: {kproto.cost_:.2f}")

Variables categóricas detectadas (11): ['estado_examen_balance_1', 'fracturas_1', 'periodo_examen_1', 'raza_etnia_4.0', "tipo_sonda_elasto_b'M'", 'raza_etnia_6.0', 'medicacion_osteoporosis_1', 'elegibilidad_balance_1', 'convulsiones_1', 'psu', 'raza_etnia_2.0']
Variables numéricas detectadas (20): ['rigidez_iqr', 'cap_iqr', 'intentos_totales_elasto', 'largo_brazo_superior_cm', 'mercurio_sangre_nmol_l', 'tiempo_seg_cond3', 'hdl_mmol_l', 'creatinina_orina_umol_l', 'hemoglobina_g_dl', 'tiempo_seg_cond1', 'peso_kg', 'volumen_plaquetario_medio', 'ancho_distribucion_eritrocitos', 'medidas_validas_elasto', 'conc_hemoglobina_media', 'tamano_hogar', 'pulso', 'plaquetas_totales', 'edad_an', 'trigliceridos_mmol_l']

Entrenando K-Prototypes... (esto puede tardar unos segundos/minutos)

--- Número de filas por clúster (K-Prototypes) ---
Clúster 0: 816 filas
Clúster 1: 30 filas
Clúster 2: 259 filas
Clúster 3: 814 filas
--------------------------------------------------
Costo del modelo: 34067.88


In [104]:
import gower
from sklearn.metrics import silhouette_score

# 1. Asegurarnos de que las categóricas sean texto para que Gower las evalúe bien
X_eval = X.copy()
for col in cat_cols:
    X_eval[col] = X_eval[col].astype(str)

# 2. Calcular la matriz de distancias (El árbitro imparcial)
print("\nCalculando matriz de Gower para evaluación...")
dist_matrix_eval = gower.gower_matrix(X_eval)

# 3. Calcular el Silhouette Score usando las etiquetas que generó K-Prototypes
score_kproto = silhouette_score(dist_matrix_eval, labels, metric='precomputed')

print(f"Silhouette Score (Gower) de K-Prototypes para k=4: {score_kproto:.4f}")


Calculando matriz de Gower para evaluación...
Silhouette Score (Gower) de K-Prototypes para k=4: 0.1708


In [105]:
df_train_target=df_train_target.copy()

In [106]:
df_train_target['clusters'] = labels
print(df_train_target.head())

# (Opcional) Ver la relación rápida entre los clústeres y la hipertensión


   hipertension  clusters
0             1         3
1             0         2
2             1         0
3             1         0
4             1         3


In [107]:
print("\n--- Relación: Clúster vs Hipertensión ---")
print(pd.crosstab(df_train_target['clusters'], df_train_target['hipertension']))


--- Relación: Clúster vs Hipertensión ---
hipertension    0    1
clusters              
0             388  428
1              12   18
2             134  125
3             515  299


In [108]:
import pandas as pd
import numpy as np
import gower
from sklearn_extra.cluster import KMedoids
from sklearn.metrics import silhouette_score

# Tu lista de columnas
columnas_genetico = [
    "estado_examen_balance_1", "rigidez_iqr", "cap_iqr", "intentos_totales_elasto",
    "fracturas_1", "largo_brazo_superior_cm", "mercurio_sangre_nmol_l", 
    "periodo_examen_1", "raza_etnia_4.0", "tiempo_seg_cond3", "hdl_mmol_l", 
    "tipo_sonda_elasto_b'M'", "raza_etnia_6.0", "medicacion_osteoporosis_1", 
    "creatinina_orina_umol_l", "hemoglobina_g_dl", "tiempo_seg_cond1", 
    "elegibilidad_balance_1", "convulsiones_1", "peso_kg", "psu", 
    "volumen_plaquetario_medio", "ancho_distribucion_eritrocitos", 
    "medidas_validas_elasto", "conc_hemoglobina_media", "tamano_hogar", 
    "pulso", "plaquetas_totales", "edad_an", "trigliceridos_mmol_l", "raza_etnia_2.0"
]

# 1. Crear el dataframe de trabajo
X = trainset[columnas_genetico].copy()

# 2. Separar dinámicamente y adaptar tipos de datos para Gower
# Gower asume que si es texto/categoría usa coincidencia, si es número usa distancias.
cat_cols = [col for col in X.columns if X[col].dtype == 'object' or X[col].nunique() <= 2]
num_cols = [col for col in X.columns if col not in cat_cols]

print(f"Variables categóricas detectadas ({len(cat_cols)}):", cat_cols)
print(f"Variables numéricas detectadas ({len(num_cols)}):", num_cols)

# CRÍTICO: Convertimos las variables binarias/categóricas a texto (string)
# Esto asegura que gower_matrix no intente calcular promedios entre 0 y 1
for col in cat_cols:
    X[col] = X[col].astype(str)

# 3. Calcular la Matriz de Distancias de Gower
print("\nCalculando matriz de Gower... (esto tomará memoria y tiempo)")
dist_matrix = gower.gower_matrix(X)

# 4. Inicializar y entrenar K-Medoids (PAM) para k=4
# Usamos metric='precomputed' porque le pasamos la matriz de Gower, no los datos crudos
print("Entrenando K-Medoids (PAM)...")
kmed = KMedoids(n_clusters=2, metric='precomputed', method='pam', random_state=42)

# Entrenamos y predecimos pasándole la matriz de distancias
labels = kmed.fit_predict(dist_matrix)

# 5. Guardar los resultados y mostrar tamaños
X['cluster'] = labels
conteos = X['cluster'].value_counts().sort_index()

print("\n--- Número de filas por clúster (K-Medoids PAM) ---")
for cluster_id, cantidad in conteos.items():
    print(f"Clúster {cluster_id}: {cantidad} filas")
print("---------------------------------------------------")

# 6. Calcular el Silhouette Score
# Al igual que K-Medoids, el Silhouette Score debe usar la matriz precalculada
score = silhouette_score(dist_matrix, labels, metric='precomputed')

print(f"\nEl Silhouette Score (basado en distancias Gower) para k=2 es: {score:.4f}")

Variables categóricas detectadas (11): ['estado_examen_balance_1', 'fracturas_1', 'periodo_examen_1', 'raza_etnia_4.0', "tipo_sonda_elasto_b'M'", 'raza_etnia_6.0', 'medicacion_osteoporosis_1', 'elegibilidad_balance_1', 'convulsiones_1', 'psu', 'raza_etnia_2.0']
Variables numéricas detectadas (20): ['rigidez_iqr', 'cap_iqr', 'intentos_totales_elasto', 'largo_brazo_superior_cm', 'mercurio_sangre_nmol_l', 'tiempo_seg_cond3', 'hdl_mmol_l', 'creatinina_orina_umol_l', 'hemoglobina_g_dl', 'tiempo_seg_cond1', 'peso_kg', 'volumen_plaquetario_medio', 'ancho_distribucion_eritrocitos', 'medidas_validas_elasto', 'conc_hemoglobina_media', 'tamano_hogar', 'pulso', 'plaquetas_totales', 'edad_an', 'trigliceridos_mmol_l']

Calculando matriz de Gower... (esto tomará memoria y tiempo)
Entrenando K-Medoids (PAM)...

--- Número de filas por clúster (K-Medoids PAM) ---
Clúster 0: 904 filas
Clúster 1: 1015 filas
---------------------------------------------------

El Silhouette Score (basado en distancias Gow

In [109]:
import pandas as pd
import numpy as np
import gower
from sklearn_extra.cluster import KMedoids
from sklearn.metrics import silhouette_score

# Tu lista de columnas
columnas_genetico = [
    "estado_examen_balance_1", "rigidez_iqr", "cap_iqr", "intentos_totales_elasto",
    "fracturas_1", "largo_brazo_superior_cm", "mercurio_sangre_nmol_l", 
    "periodo_examen_1", "raza_etnia_4.0", "tiempo_seg_cond3", "hdl_mmol_l", 
    "tipo_sonda_elasto_b'M'", "raza_etnia_6.0", "medicacion_osteoporosis_1", 
    "creatinina_orina_umol_l", "hemoglobina_g_dl", "tiempo_seg_cond1", 
    "elegibilidad_balance_1", "convulsiones_1", "peso_kg", "psu", 
    "volumen_plaquetario_medio", "ancho_distribucion_eritrocitos", 
    "medidas_validas_elasto", "conc_hemoglobina_media", "tamano_hogar", 
    "pulso", "plaquetas_totales", "edad_an", "trigliceridos_mmol_l", "raza_etnia_2.0"
]

# 1. Crear el dataframe de trabajo
X = trainset[columnas_genetico].copy()

# 2. Separar dinámicamente y adaptar tipos de datos para Gower
# Gower asume que si es texto/categoría usa coincidencia, si es número usa distancias.
cat_cols = [col for col in X.columns if X[col].dtype == 'object' or X[col].nunique() <= 2]
num_cols = [col for col in X.columns if col not in cat_cols]

print(f"Variables categóricas detectadas ({len(cat_cols)}):", cat_cols)
print(f"Variables numéricas detectadas ({len(num_cols)}):", num_cols)

# CRÍTICO: Convertimos las variables binarias/categóricas a texto (string)
# Esto asegura que gower_matrix no intente calcular promedios entre 0 y 1
for col in cat_cols:
    X[col] = X[col].astype(str)

# 3. Calcular la Matriz de Distancias de Gower
print("\nCalculando matriz de Gower... (esto tomará memoria y tiempo)")
dist_matrix = gower.gower_matrix(X)

# 4. Inicializar y entrenar K-Medoids (PAM) para k=4
# Usamos metric='precomputed' porque le pasamos la matriz de Gower, no los datos crudos
print("Entrenando K-Medoids (PAM)...")
kmed = KMedoids(n_clusters=3, metric='precomputed', method='pam', random_state=42)

# Entrenamos y predecimos pasándole la matriz de distancias
labels = kmed.fit_predict(dist_matrix)

# 5. Guardar los resultados y mostrar tamaños
X['cluster'] = labels
conteos = X['cluster'].value_counts().sort_index()

print("\n--- Número de filas por clúster (K-Medoids PAM) ---")
for cluster_id, cantidad in conteos.items():
    print(f"Clúster {cluster_id}: {cantidad} filas")
print("---------------------------------------------------")

# 6. Calcular el Silhouette Score
# Al igual que K-Medoids, el Silhouette Score debe usar la matriz precalculada
score = silhouette_score(dist_matrix, labels, metric='precomputed')

print(f"\nEl Silhouette Score (basado en distancias Gower) para k=2 es: {score:.4f}")

Variables categóricas detectadas (11): ['estado_examen_balance_1', 'fracturas_1', 'periodo_examen_1', 'raza_etnia_4.0', "tipo_sonda_elasto_b'M'", 'raza_etnia_6.0', 'medicacion_osteoporosis_1', 'elegibilidad_balance_1', 'convulsiones_1', 'psu', 'raza_etnia_2.0']
Variables numéricas detectadas (20): ['rigidez_iqr', 'cap_iqr', 'intentos_totales_elasto', 'largo_brazo_superior_cm', 'mercurio_sangre_nmol_l', 'tiempo_seg_cond3', 'hdl_mmol_l', 'creatinina_orina_umol_l', 'hemoglobina_g_dl', 'tiempo_seg_cond1', 'peso_kg', 'volumen_plaquetario_medio', 'ancho_distribucion_eritrocitos', 'medidas_validas_elasto', 'conc_hemoglobina_media', 'tamano_hogar', 'pulso', 'plaquetas_totales', 'edad_an', 'trigliceridos_mmol_l']

Calculando matriz de Gower... (esto tomará memoria y tiempo)
Entrenando K-Medoids (PAM)...

--- Número de filas por clúster (K-Medoids PAM) ---
Clúster 0: 478 filas
Clúster 1: 674 filas
Clúster 2: 767 filas
---------------------------------------------------

El Silhouette Score (basa

In [111]:
import pandas as pd
import numpy as np
import gower
from sklearn_extra.cluster import KMedoids
from sklearn.metrics import silhouette_score

# Tu lista de columnas
columnas_genetico = [
    "estado_examen_balance_1", "rigidez_iqr", "cap_iqr", "intentos_totales_elasto",
    "fracturas_1", "largo_brazo_superior_cm", "mercurio_sangre_nmol_l", 
    "periodo_examen_1", "raza_etnia_4.0", "tiempo_seg_cond3", "hdl_mmol_l", 
    "tipo_sonda_elasto_b'M'", "raza_etnia_6.0", "medicacion_osteoporosis_1", 
    "creatinina_orina_umol_l", "hemoglobina_g_dl", "tiempo_seg_cond1", 
    "elegibilidad_balance_1", "convulsiones_1", "peso_kg", "psu", 
    "volumen_plaquetario_medio", "ancho_distribucion_eritrocitos", 
    "medidas_validas_elasto", "conc_hemoglobina_media", "tamano_hogar", 
    "pulso", "plaquetas_totales", "edad_an", "trigliceridos_mmol_l", "raza_etnia_2.0"
]

# 1. Crear el dataframe de trabajo
X = trainset[columnas_genetico].copy()

# 2. Separar dinámicamente y adaptar tipos de datos para Gower
# Gower asume que si es texto/categoría usa coincidencia, si es número usa distancias.
cat_cols = [col for col in X.columns if X[col].dtype == 'object' or X[col].nunique() <= 2]
num_cols = [col for col in X.columns if col not in cat_cols]

print(f"Variables categóricas detectadas ({len(cat_cols)}):", cat_cols)
print(f"Variables numéricas detectadas ({len(num_cols)}):", num_cols)

# CRÍTICO: Convertimos las variables binarias/categóricas a texto (string)
# Esto asegura que gower_matrix no intente calcular promedios entre 0 y 1
for col in cat_cols:
    X[col] = X[col].astype(str)

# 3. Calcular la Matriz de Distancias de Gower
print("\nCalculando matriz de Gower... (esto tomará memoria y tiempo)")
dist_matrix = gower.gower_matrix(X)

# 4. Inicializar y entrenar K-Medoids (PAM) para k=4
# Usamos metric='precomputed' porque le pasamos la matriz de Gower, no los datos crudos
print("Entrenando K-Medoids (PAM)...")
kmed = KMedoids(n_clusters=4, metric='precomputed', method='pam', random_state=42)

# Entrenamos y predecimos pasándole la matriz de distancias
labels = kmed.fit_predict(dist_matrix)

# 5. Guardar los resultados y mostrar tamaños
X['cluster'] = labels
conteos = X['cluster'].value_counts().sort_index()

print("\n--- Número de filas por clúster (K-Medoids PAM) ---")
for cluster_id, cantidad in conteos.items():
    print(f"Clúster {cluster_id}: {cantidad} filas")
print("---------------------------------------------------")

# 6. Calcular el Silhouette Score
# Al igual que K-Medoids, el Silhouette Score debe usar la matriz precalculada
score = silhouette_score(dist_matrix, labels, metric='precomputed')

print(f"\nEl Silhouette Score (basado en distancias Gower) para k=2 es: {score:.4f}")

Variables categóricas detectadas (11): ['estado_examen_balance_1', 'fracturas_1', 'periodo_examen_1', 'raza_etnia_4.0', "tipo_sonda_elasto_b'M'", 'raza_etnia_6.0', 'medicacion_osteoporosis_1', 'elegibilidad_balance_1', 'convulsiones_1', 'psu', 'raza_etnia_2.0']
Variables numéricas detectadas (20): ['rigidez_iqr', 'cap_iqr', 'intentos_totales_elasto', 'largo_brazo_superior_cm', 'mercurio_sangre_nmol_l', 'tiempo_seg_cond3', 'hdl_mmol_l', 'creatinina_orina_umol_l', 'hemoglobina_g_dl', 'tiempo_seg_cond1', 'peso_kg', 'volumen_plaquetario_medio', 'ancho_distribucion_eritrocitos', 'medidas_validas_elasto', 'conc_hemoglobina_media', 'tamano_hogar', 'pulso', 'plaquetas_totales', 'edad_an', 'trigliceridos_mmol_l']

Calculando matriz de Gower... (esto tomará memoria y tiempo)
Entrenando K-Medoids (PAM)...

--- Número de filas por clúster (K-Medoids PAM) ---
Clúster 0: 477 filas
Clúster 1: 502 filas
Clúster 2: 566 filas
Clúster 3: 374 filas
---------------------------------------------------

El S

In [112]:
df_train_target=df_train_target.copy()

In [113]:
df_train_target['clusters'] = labels
print(df_train_target.head())

# (Opcional) Ver la relación rápida entre los clústeres y la hipertensión


   hipertension  clusters
0             1         2
1             0         2
2             1         2
3             1         1
4             1         3


In [114]:
print("\n--- Relación: Clúster vs Hipertensión ---")
print(pd.crosstab(df_train_target['clusters'], df_train_target['hipertension']))


--- Relación: Clúster vs Hipertensión ---
hipertension    0    1
clusters              
0             289  188
1             266  236
2             298  268
3             196  178
